### DESE (Department of Elementary and Secondary Education) in Massachusetts 

Este dataset contiene información sobre el sistema educativo en Massachusetts, organizado a nivel de distritos y escuelas. Permite analizar relaciones entre desempeño académico, gasto educativo y evaluaciones del personal.

---

### Estructura de la base de datos

El sistema se organiza en torno a distritos escolares, que contienen múltiples escuelas y concentran información financiera y evaluaciones.

#### districts
Contiene información general de los distritos escolares.
- `id`: identificador del distrito  
- `name`: nombre del distrito  
- `type`: tipo de distrito (público o charter)  

---

#### schools
Representa las escuelas dentro de cada distrito.
- `id`: identificador de la escuela  
- `district_id`: distrito al que pertenece  
- `name`: nombre de la escuela  
- `type`: tipo de escuela  

---

#### graduation_rates
Contiene métricas de desempeño académico por escuela.
- `school_id`: escuela asociada  
- `graduated`: porcentaje de graduación  
- `dropped`: porcentaje de abandono escolar  

---

#### expenditures
Información financiera a nivel distrito.
- `district_id`: distrito asociado  
- `pupils`: número de estudiantes  
- `per_pupil_expenditure`: gasto por alumno  

---

#### staff_evaluations
Evaluación del personal educativo por distrito.
- `district_id`: distrito asociado  
- `exemplary`: porcentaje de desempeño ejemplar  
- `unsatisfactory`: porcentaje de desempeño insatisfactorio  

---

### Relaciones clave

- Un **distrito** contiene múltiples **escuelas**  
- Cada **escuela** reporta una métrica de **graduación**  
- Los **distritos** pueden no hacer **gasto** o múltiples y asimismo no recibir **evaluación del personal** o recibir múltiples  



In [ ]:
# Conexión a la base de datos

import sqlite3
import pandas as pd

conn = sqlite3.connect("dese.db")

In [13]:
def mostrar_tabla(nombre):
    query = f"SELECT*FROM {nombre} LIMIT 5;"
    return pd.read_sql_query(query,conn)

In [8]:
mostrar_tabla("districts")

,id,name,type,city,state,zip
0,1,Abby Kelley Foster Charter Public (District),Charter District,Worcester,MA,01606
1,2,Abington,Public School District,Abington,MA,02351
2,3,Academy Of the Pacific Rim Charter Public (Dis...,Charter District,Hyde Park,MA,02136
3,4,Acton (non-op),Public School District,Acton,MA,01720
4,5,Acton-Boxborough,Public School District,Acton,MA,01720


In [9]:
mostrar_tabla("schools")

,id,district_id,name,type,city,state,zip
0,1,1,Abby Kelley Foster Charter Public School,Charter School,Worcester,MA,01606
1,2,2,Abington Early Education Program,Public School,Abington,MA,02351
2,3,2,Abington High,Public School,Abington,MA,02351
3,4,2,Abington Middle School,Public School,Abington,MA,02351
4,5,2,Beaver Brook Elementary,Public School,Abington,MA,02351


In [10]:
mostrar_tabla("graduation_rates")

,id,school_id,graduated,dropped,excluded
0,1,1,98.8,1.2,0
1,2,3,93.8,3.7,0
2,3,7,93.2,0.0,0
3,4,8,98.8,0.7,0
4,5,19,98.6,0.0,0


In [11]:
mostrar_tabla("expenditures")

,id,district_id,pupils,per_pupil_expenditure
0,1,1,1424,16853.85
1,2,2,2244,16143.23
2,3,3,539,21216.47
3,4,5,5345,18957.61
4,5,6,1235,14908.37


In [12]:
mostrar_tabla("staff_evaluations")

,id,district_id,evaluated,exemplary,proficient,needs_improvement,unsatisfactory
0,1,1,100.0,5.3,94.7,0.0,0.0
1,2,2,98.2,15.4,83.3,0.6,0.6
2,3,3,100.0,NaN,NaN,NaN,NaN
3,4,5,96.6,13.1,85.9,1.0,0.0
4,5,6,98.8,10.7,82.1,6.0,1.2



### **FILTROS Y AGRUPACIONES**

Ciudades con 3 o menos escuelas públicas

**WHERE:** se utiliza para aplicar condiciones a nivel de fila

**HAVING:** se utiliza para aplicar condiciones a nivel de grupo

In [16]:
query = """

SELECT "city",COUNT("type") AS "Public Schools" 
FROM "schools"
WHERE "type"='Public School'
GROUP BY "city"
HAVING "Public Schools" <= 3
ORDER BY "Public Schools" DESC, "city" ASC;

"""
pd.read_sql_query(query,conn)

,city,Public Schools
0,Ashburnham,3
1,Athol,3
2,Barre,3
3,Charlestown,3
4,Chestnut Hill,3
...,...,...
196,West Warren,1
197,West Yarmouth,1
198,Whately,1
199,Williamsburg,1


### **Subconsultas**

Escuelas con un 100% de tasa de graduación

**IN:** se utiliza cuando la subconsulta devuelve múltiples valores

In [17]:
query = """

SELECT "name" FROM "schools" WHERE "id" IN (
    SELECT "school_id" FROM "graduation_rates" 
    WHERE "graduated"=100
);

"""
pd.read_sql_query(query,conn)

,name
0,Tahanto Regional High
1,O'Bryant School of Math & Science
2,Cohasset High School
3,Global Learning Charter Public School
4,Bromfield
5,Pioneer Valley Regional
6,Sizer School: A North Central Charter Essentia...
7,Upper Cape Cod Vocational Technical
8,Weston High
